# 04 — Module 1: Fidelity (faithfulness) of XAI explanations

Does removing (or keeping only) the features an explanation flags as
important actually change the model's prediction? Three complementary
metrics:

1. **Comprehensiveness ↑** — drop in predicted fraud probability when the
   top-k features are zeroed out.
2. **Sufficiency ↓** — drop when only the top-k features are kept.
3. **Infidelity ↓** — mean squared error between the attribution's
   linear prediction ``I · attr`` and the model's actual change
   ``f(x) - f(x - I)`` under Gaussian input perturbations.

**Coverage (14 configurations):**

- ML: {RF, LightGBM} × {SHAP, LIME} × {Elliptic, Ethereum} = 8 rows
- GNN: {Temporal-GCN, GraphSAGE} × {GNNExplainer, IG, GraphLIME} × Elliptic = 6 rows

**Prerequisite:** notebooks 02a, 02b, 03a, 03b have been executed.
**Training performed in this notebook:** none.

**Outputs:**

- `experiments/results/module1_fidelity_results.csv`
- `experiments/results/module1_fidelity_summary_k5.csv`
- `experiments/figures/module1_comprehensiveness_vs_k.png`
- `experiments/figures/module1_sufficiency_vs_k.png`
- `experiments/figures/module1_infidelity_bar.png`
- `experiments/figures/module1_heatmap_{elliptic,ethereum}.png`


In [ ]:
import os
from datetime import datetime

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
import warnings
warnings.filterwarnings('ignore')

from xai_blockchain_framework.config import PATHS, set_seed
from xai_blockchain_framework.models import (
    TemporalGCN, GraphSAGEModel, get_device, make_fraud_predict_fn,
)
from xai_blockchain_framework.metrics import (
    comprehensiveness, sufficiency, infidelity,
    gnn_comprehensiveness, gnn_sufficiency, gnn_infidelity,
)

set_seed(42)

DEVICE = get_device()
RESULTS_PATH = str(PATHS.results_dir)
FIGURES_PATH = str(PATHS.figures_dir)
MODELS_PATH = str(PATHS.models_dir)
PROCESSED_PATH = str(PATHS.data_processed)
for p in [RESULTS_PATH, FIGURES_PATH, PROCESSED_PATH]:
    os.makedirs(p, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')

K_VALUES = [1, 3, 5, 10]
N_PERT = 50
SIGMA = 0.1
RNG = np.random.default_rng(42)

print("=" * 70)
print("MODULE 1  FIDELITY")
print("=" * 70)
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"k_values={K_VALUES}, n_perturbations={N_PERT}, sigma={SIGMA}")
print(f"Device: {DEVICE}")


## 1. Load models, splits, and pre-computed XAI attributions


In [ ]:
print("\n" + "=" * 70)
print("1. Loading models and explanations")
print("=" * 70)

# --- ML models ---
rf_ell = joblib.load(os.path.join(MODELS_PATH, 'elliptic_rf.joblib'))
lgb_ell = joblib.load(os.path.join(MODELS_PATH, 'elliptic_lgb.joblib'))
rf_eth = joblib.load(os.path.join(MODELS_PATH, 'ethereum_rf.joblib'))
lgb_eth = joblib.load(os.path.join(MODELS_PATH, 'ethereum_lgb.joblib'))
print("  ML models loaded")

# --- Splits ---
ell = np.load(os.path.join(PROCESSED_PATH, 'elliptic_splits.npz'))
eth = np.load(os.path.join(PROCESSED_PATH, 'ethereum_splits.npz'))
X_test_ell, y_test_ell = ell['X_test'], ell['y_test']
X_test_eth, y_test_eth = eth['X_test'], eth['y_test']

# --- Eval indices (from 03a) ---
eval_idx_ell = np.load(os.path.join(RESULTS_PATH, 'xai_eval_indices_elliptic.npy'))
eval_idx_eth = np.load(os.path.join(RESULTS_PATH, 'xai_eval_indices_ethereum.npy'))

# --- SHAP values (from 03a) ---
shap_rf_ell = np.load(os.path.join(RESULTS_PATH, 'shap_values_rf_elliptic.npy'))
shap_lgb_ell = np.load(os.path.join(RESULTS_PATH, 'shap_values_lgb_elliptic.npy'))
shap_rf_eth = np.load(os.path.join(RESULTS_PATH, 'shap_values_rf_ethereum.npy'))
shap_lgb_eth = np.load(os.path.join(RESULTS_PATH, 'shap_values_lgb_ethereum.npy'))

# --- LIME attributions (from 03a) ---
lime_rf_ell = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_rf_elliptic.npy'))
lime_lgb_ell = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_lgb_elliptic.npy'))
lime_rf_eth = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_rf_ethereum.npy'))
lime_lgb_eth = np.load(os.path.join(RESULTS_PATH, 'lime_attrs_lgb_ethereum.npy'))
N_LIME = lime_rf_ell.shape[0]
print(f"  SHAP / LIME loaded (SHAP={shap_rf_ell.shape}, N_LIME={N_LIME})")

# --- GNN attributions (from 03b) ---
gnn_node_idx = np.load(os.path.join(RESULTS_PATH, 'gnn_eval_node_indices.npy'))
gnnexp_tgcn = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_gnnexp_tgcn.npy'))
gnnexp_sage = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_gnnexp_sage.npy'))
ig_tgcn = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_ig_tgcn.npy'))
ig_sage = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_ig_sage.npy'))
glime_tgcn = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_glime_tgcn.npy'))
glime_sage = np.load(os.path.join(RESULTS_PATH, 'gnn_attrs_glime_sage.npy'))
print(f"  GNN attributions loaded ({gnnexp_tgcn.shape[0]} nodes x 6 configs)")

# --- GNN models and graph (from 02b) ---
n_feat = shap_rf_ell.shape[1]
tgcn = TemporalGCN(in_c=n_feat).to(DEVICE)
tgcn.load_state_dict(torch.load(os.path.join(MODELS_PATH, 'elliptic_temporal_gcn.pt'), map_location=DEVICE))
tgcn.eval()
sage = GraphSAGEModel(in_c=n_feat).to(DEVICE)
sage.load_state_dict(torch.load(os.path.join(MODELS_PATH, 'elliptic_graphsage.pt'), map_location=DEVICE))
sage.eval()

edge_index = torch.load(os.path.join(PROCESSED_PATH, 'elliptic_edge_index.pt'))
ts_tensor = torch.tensor(ell['ts_all'], dtype=torch.float32).to(DEVICE)
graph_data = Data(
    x=torch.tensor(ell['X_all_n'], dtype=torch.float32),
    y=torch.tensor(ell['y_all'], dtype=torch.long),
    edge_index=edge_index,
    train_mask=torch.tensor(ell['train_mask']),
    val_mask=torch.tensor(ell['val_mask']),
    test_mask=torch.tensor(ell['test_mask']),
).to(DEVICE)
print("  GNN models and graph loaded")

# --- Prediction wrappers ---
pred_rf_ell = make_fraud_predict_fn(rf_ell, X_test_ell.shape[1])
pred_lgb_ell = make_fraud_predict_fn(lgb_ell, X_test_ell.shape[1])
pred_rf_eth = make_fraud_predict_fn(rf_eth, X_test_eth.shape[1])
pred_lgb_eth = make_fraud_predict_fn(lgb_eth, X_test_eth.shape[1])
print("  predict_proba wrappers created")


## 2. Evaluate fidelity on ML baselines

Aggregated across 200 (SHAP) or 100 (LIME) balanced test instances per
dataset. Each configuration produces one (Comp, Suff, Infid) triplet per
value of k in {1, 3, 5, 10}.


In [ ]:
print("\n" + "=" * 70)
print("2. ML baselines")
print("=" * 70)

all_results: list[dict] = []

ML_SHAP = [
    ('RF',  'Elliptic', pred_rf_ell,  X_test_ell, eval_idx_ell, shap_rf_ell),
    ('LGB', 'Elliptic', pred_lgb_ell, X_test_ell, eval_idx_ell, shap_lgb_ell),
    ('RF',  'Ethereum', pred_rf_eth,  X_test_eth, eval_idx_eth, shap_rf_eth),
    ('LGB', 'Ethereum', pred_lgb_eth, X_test_eth, eval_idx_eth, shap_lgb_eth),
]
ML_LIME = [
    ('RF',  'Elliptic', pred_rf_ell,  X_test_ell, eval_idx_ell[:N_LIME], lime_rf_ell),
    ('LGB', 'Elliptic', pred_lgb_ell, X_test_ell, eval_idx_ell[:N_LIME], lime_lgb_ell),
    ('RF',  'Ethereum', pred_rf_eth,  X_test_eth, eval_idx_eth[:N_LIME], lime_rf_eth),
    ('LGB', 'Ethereum', pred_lgb_eth, X_test_eth, eval_idx_eth[:N_LIME], lime_lgb_eth),
]

def eval_ml(configs, xai_name):
    for model_name, dataset, predict_fn, X, indices, attrs in configs:
        label = f"{model_name}-{dataset}-{xai_name}"
        print(f"\n--- {label} ({len(indices)} instances) ---")
        infid = infidelity(predict_fn, X, attrs, indices,
                           n_perturbations=N_PERT, sigma=SIGMA,
                           rng=np.random.default_rng(42))
        for k in K_VALUES:
            comp = comprehensiveness(predict_fn, X, attrs, indices, k=k)
            suff = sufficiency(predict_fn, X, attrs, indices, k=k)
            all_results.append({
                'Model': model_name, 'Dataset': dataset, 'XAI': xai_name,
                'Type': 'ML', 'k': k,
                'Comprehensiveness': comp, 'Sufficiency': suff, 'Infidelity': infid,
            })
        print(f"  Comp@5={all_results[-2]['Comprehensiveness']:.4f}"
              f"  Suff@5={all_results[-2]['Sufficiency']:.4f}"
              f"  Infid={infid:.6f}")

eval_ml(ML_SHAP, 'SHAP')
eval_ml(ML_LIME, 'LIME')


## 3. Evaluate fidelity on GNN models (Elliptic)

Per-node forward passes through the Temporal-GCN and GraphSAGE models;
attributions come from GNNExplainer, Integrated Gradients and a
feature-perturbation GraphLIME.


In [ ]:
print("\n" + "=" * 70)
print("3. GNN models")
print("=" * 70)

GNN_CONFIGS = [
    ('T-GCN', tgcn, 'GNNExp',    gnnexp_tgcn),
    ('T-GCN', tgcn, 'IG',        ig_tgcn),
    ('T-GCN', tgcn, 'GraphLIME', glime_tgcn),
    ('SAGE',  sage, 'GNNExp',    gnnexp_sage),
    ('SAGE',  sage, 'IG',        ig_sage),
    ('SAGE',  sage, 'GraphLIME', glime_sage),
]

for model_name, model, xai_name, attrs in GNN_CONFIGS:
    label = f"{model_name}-{xai_name}"
    print(f"\n--- {label} ({len(gnn_node_idx)} nodes) ---")
    has_ts = hasattr(model, 'time_enc')

    def forward(x_in):
        return model(x_in, graph_data.edge_index, ts=ts_tensor) if has_ts \
            else model(x_in, graph_data.edge_index)

    comp_all = {k: [] for k in K_VALUES}
    suff_all = {k: [] for k in K_VALUES}
    infid_all: list[float] = []
    rng = np.random.default_rng(42)

    for i, nid in enumerate(gnn_node_idx):
        nid = int(nid)
        expl = attrs[i]
        comp_k = gnn_comprehensiveness(forward, graph_data.x, nid, expl, k_values=K_VALUES)
        suff_k = gnn_sufficiency(forward, graph_data.x, nid, expl, k_values=K_VALUES)
        infid_i = gnn_infidelity(forward, graph_data.x, nid, expl,
                                 n_perturbations=N_PERT, sigma=SIGMA, rng=rng)
        for k in K_VALUES:
            if k in comp_k:
                comp_all[k].append(comp_k[k])
            if k in suff_k:
                suff_all[k].append(suff_k[k])
        infid_all.append(infid_i)
        if (i + 1) % 25 == 0:
            print(f"  {i + 1}/{len(gnn_node_idx)}")

    comp_avg = {k: float(np.mean(v)) for k, v in comp_all.items() if v}
    suff_avg = {k: float(np.mean(v)) for k, v in suff_all.items() if v}
    infid_avg = float(np.mean(infid_all))
    print(f"  Comp@5={comp_avg.get(5,0):.4f}  Suff@5={suff_avg.get(5,0):.4f}  Infid={infid_avg:.6f}")

    for k in K_VALUES:
        all_results.append({
            'Model': model_name, 'Dataset': 'Elliptic', 'XAI': xai_name,
            'Type': 'GNN', 'k': k,
            'Comprehensiveness': comp_avg.get(k, np.nan),
            'Sufficiency': suff_avg.get(k, np.nan),
            'Infidelity': infid_avg,
        })


## 4. Summary table (k=5)


In [ ]:
results_df = pd.DataFrame(all_results)
summary = results_df[results_df['k'] == 5][
    ['Type', 'Model', 'Dataset', 'XAI', 'Comprehensiveness', 'Sufficiency', 'Infidelity']
].reset_index(drop=True)
print(summary.to_string(index=False))


## 5. Visualizations


In [ ]:
print("\n" + "=" * 70)
print("5. Visualizations")
print("=" * 70)

COLORS_ML = {
    'RF-SHAP': '#2ecc71', 'RF-LIME': '#27ae60',
    'LGB-SHAP': '#3498db', 'LGB-LIME': '#2980b9',
}
COLORS_GNN = {
    'T-GCN-GNNExp': '#e74c3c', 'T-GCN-IG': '#c0392b', 'T-GCN-GraphLIME': '#f39c12',
    'SAGE-GNNExp': '#9b59b6',  'SAGE-IG': '#8e44ad',   'SAGE-GraphLIME': '#e67e22',
}

def get_color(row):
    key = f"{row['Model']}-{row['XAI']}"
    return COLORS_ML.get(key, '#95a5a6') if row['Type'] == 'ML' else COLORS_GNN.get(key, '#95a5a6')

def _vs_k_plot(metric, marker, direction, out_name):
    fig, ax = plt.subplots(figsize=(12, 7))
    for _, grp in results_df.groupby(['Model', 'XAI', 'Dataset']):
        g = grp.sort_values('k')
        lbl = f"{g['Model'].iloc[0]}-{g['XAI'].iloc[0]}-{g['Dataset'].iloc[0]}"
        ls = '-' if g['Type'].iloc[0] == 'ML' else '--'
        color = get_color(g.iloc[0])
        ax.plot(g['k'], g[metric], marker=marker, label=lbl, linestyle=ls,
                markersize=6, linewidth=2, color=color)
    ax.set_xlabel('k (number of features)', fontsize=12)
    ax.set_ylabel(f'{metric} {direction}', fontsize=12)
    ax.set_title(f'Module 1  {metric} vs k', fontsize=14, fontweight='bold')
    ax.legend(fontsize=8, ncol=2, loc='upper left', bbox_to_anchor=(1.02, 1))
    ax.grid(True, alpha=0.3)
    ax.set_xticks(K_VALUES)
    plt.tight_layout()
    path = os.path.join(FIGURES_PATH, out_name)
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  wrote {out_name}")

_vs_k_plot('Comprehensiveness', 'o', '', 'module1_comprehensiveness_vs_k.png')
_vs_k_plot('Sufficiency', 's', '', 'module1_sufficiency_vs_k.png')

fig, ax = plt.subplots(figsize=(14, 7))
infid = results_df[results_df['k'] == 5].drop_duplicates(['Model', 'XAI', 'Dataset'])
labels = [f"{r['Model']}\n{r['XAI']}\n{r['Dataset']}" for _, r in infid.iterrows()]
colors_bar = [get_color(r) for _, r in infid.iterrows()]
bars = ax.bar(range(len(infid)), infid['Infidelity'].values, alpha=0.85,
              color=colors_bar, edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(infid)))
ax.set_xticklabels(labels, fontsize=9, rotation=45, ha='right')
ax.set_ylabel('Infidelity', fontsize=12)
ax.set_title('Module 1  Infidelity Score (k=5)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, infid['Infidelity'].values):
    ax.annotate(f'{val:.4f}', xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, 'module1_infidelity_bar.png'), dpi=150, bbox_inches='tight')
plt.show()
print("  wrote module1_infidelity_bar.png")

def _heatmap(dataset, out_name):
    fig, ax = plt.subplots(figsize=(10, 6))
    sub = results_df[(results_df['Dataset'] == dataset) & (results_df['k'] == 5)]
    if sub.empty:
        print(f"  (skipped {out_name}: no rows for {dataset})")
        return
    pivot = sub.pivot_table(index='Model', columns='XAI', values='Comprehensiveness')
    im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto', vmin=0, vmax=0.8)
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, fontsize=11)
    ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index, fontsize=11)
    for r in range(len(pivot.index)):
        for c in range(len(pivot.columns)):
            v = pivot.values[r, c]
            if not np.isnan(v):
                ax.text(c, r, f'{v:.3f}', ha='center', va='center', fontsize=12, fontweight='bold')
    ax.set_title(f'{dataset}  Comprehensiveness (k=5)', fontsize=14, fontweight='bold')
    plt.colorbar(im, ax=ax, label='Comprehensiveness')
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_PATH, out_name), dpi=150, bbox_inches='tight')
    plt.show()
    print(f"  wrote {out_name}")

_heatmap('Elliptic', 'module1_heatmap_elliptic.png')
_heatmap('Ethereum', 'module1_heatmap_ethereum.png')


## 6. Save results


In [ ]:
results_df.to_csv(os.path.join(RESULTS_PATH, 'module1_fidelity_results.csv'), index=False)
summary.to_csv(os.path.join(RESULTS_PATH, 'module1_fidelity_summary_k5.csv'), index=False)

print("""
Files saved:
  module1_fidelity_results.csv          (full long-format table)
  module1_fidelity_summary_k5.csv       (k=5 cross-configuration summary)
  module1_comprehensiveness_vs_k.png
  module1_sufficiency_vs_k.png
  module1_infidelity_bar.png
  module1_heatmap_{elliptic,ethereum}.png

Interpretation:
  Comprehensiveness (up)   model depends on the features the explanation flags.
  Sufficiency (down)       the top-k features alone reproduce the prediction.
  Infidelity (down)        the linear attribution matches the model's response.

14 configurations evaluated (8 ML + 6 GNN).
""")
